# 🚗 Automobile Market Analytics — Fiyat Tahmin Modeli (Regresyon)

Bu projede, ikinci el otomobil pazarına ait **5.500 araç** ve **18 özellik** içeren bir veri seti kullanılarak araçların **satış fiyatını (Selling_Price)** tahmin etmeye yönelik bir regresyon çalışması yapılmaktadır.

## 📌 Veri Seti Hakkında
Veri setinde araçlara ait aşağıdaki bilgiler yer almaktadır:
- **Kategorik değişkenler:** Make, Model, Fuel_Type, Transmission, Service_History, Color, Body_Type, Drivetrain, Location
- **Sayısal değişkenler:** Year, Engine_Size, Mileage, Horsepower, Torque, Owners, Accident_History, Fuel_Efficiency
- **Hedef değişken:** Selling_Price

## 🔧 Uygulanan Adımlar
1. Eksik veri analizi ve doldurma (kategorik → mod, sayısal → medyan)
2. Aykırı değerlerin temizlenmesi
3. Keşifsel veri analizi (value_counts ile dağılım incelemesi)
4. One-Hot Encoding (kategorik) + Standart Ölçekleme (sayısal)
5. Eğitim/Test ayrımı (%75 / %25)
6. Farklı regresyon modellerinin denenmesi ve GridSearchCV ile hiperparametre optimizasyonu:
   - Linear Regression
   - Polynomial Regression
   - Lasso / Ridge / ElasticNet (CV)
   - Support Vector Regression (SVR)
   - K-Nearest Neighbors Regressor
   - Decision Tree Regressor
7. Modellerin MAE, MSE ve R² skorlarına göre karşılaştırılması

---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
df=pd.read_csv("4-)automobile_dataset.csv")
df.head()

,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
0,Mercedes-Benz,GLE,2024,Petrol,Automatic,2.3,100,186.0,196.0,1,0.0,NaN,Silver,SUV,AWD,30.0,IL,64140
1,Hyundai,Tucson,2008,Petrol,Automatic,2.3,387035,189.0,190.0,4,0.0,NaN,Blue,SUV,FWD,35.0,FL,500
2,Volkswagen,Golf,2021,Hybrid,NaN,1.9,46054,158.0,153.0,1,0.0,Partial Service,Gray,Hatchback,AWD,43.0,NY,16429
3,Chevrolet,Tahoe,2005,Petrol,NaN,2.2,141302,169.0,165.0,5,1.0,No Service,Blue,SUV,AWD,NaN,FL,2199
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792


In [3]:
df.isnull().sum()

Make                  0
Model                 0
Year                  0
Fuel_Type             0
Transmission        827
Engine_Size         844
Mileage               0
Horsepower          800
Torque              799
Owners                0
Accident_History    812
Service_History     855
Color               803
Body_Type             0
Drivetrain            0
Fuel_Efficiency     794
Location            782
Selling_Price         0
dtype: int64

In [4]:
df.shape

(5500, 18)

In [5]:
cat_col=["Transmission", "Accident_History", "Service_History", "Location", "Color"]

for i in cat_col:
    df[i]=df[i].fillna(df[i].mode()[0])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Make              5500 non-null   object 
 1   Model             5500 non-null   object 
 2   Year              5500 non-null   int64  
 3   Fuel_Type         5500 non-null   object 
 4   Transmission      5500 non-null   object 
 5   Engine_Size       4656 non-null   float64
 6   Mileage           5500 non-null   int64  
 7   Horsepower        4700 non-null   float64
 8   Torque            4701 non-null   float64
 9   Owners            5500 non-null   int64  
 10  Accident_History  5500 non-null   float64
 11  Service_History   5500 non-null   object 
 12  Color             5500 non-null   object 
 13  Body_Type         5500 non-null   object 
 14  Drivetrain        5500 non-null   object 
 15  Fuel_Efficiency   4706 non-null   float64
 16  Location          5500 non-null   object 


In [7]:
num_col=["Engine_Size", "Horsepower", "Torque", "Fuel_Efficiency"]

for i in num_col:
    df[i]=df[i].fillna(df[i].median())

In [8]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Make              5500 non-null   object 
 1   Model             5500 non-null   object 
 2   Year              5500 non-null   int64  
 3   Fuel_Type         5500 non-null   object 
 4   Transmission      5500 non-null   object 
 5   Engine_Size       5500 non-null   float64
 6   Mileage           5500 non-null   int64  
 7   Horsepower        5500 non-null   float64
 8   Torque            5500 non-null   float64
 9   Owners            5500 non-null   int64  
 10  Accident_History  5500 non-null   float64
 11  Service_History   5500 non-null   object 
 12  Color             5500 non-null   object 
 13  Body_Type         5500 non-null   object 
 14  Drivetrain        5500 non-null   object 
 15  Fuel_Efficiency   5500 non-null   float64
 16  Location          5500 non-null   object 


Make                0
Model               0
Year                0
Fuel_Type           0
Transmission        0
Engine_Size         0
Mileage             0
Horsepower          0
Torque              0
Owners              0
Accident_History    0
Service_History     0
Color               0
Body_Type           0
Drivetrain          0
Fuel_Efficiency     0
Location            0
Selling_Price       0
dtype: int64

In [9]:
for i in df.columns:
    print(df[i].value_counts())
    print("-------------------")

Make
Hyundai          579
Ford             579
Volkswagen       569
Chevrolet        556
Honda            553
Nissan           542
Mercedes-Benz    541
Toyota           531
Audi             528
BMW              522
Name: count, dtype: int64
-------------------
Model
GLE           166
Elantra       163
Passat        156
Malibu        156
Civic         156
Explorer      153
Silverado     152
Santa Fe      150
Escape        150
A4            149
Mustang       149
Tiguan        148
Sentra        146
5 Series      142
Rogue         140
RAV4          140
Q7            138
Accord        137
Atlas         136
Sonata        135
Camry         135
3 Series      134
CR-V          134
GLC           133
Corolla       133
X5            132
E-Class       132
Tucson        131
Pathfinder    129
A6            129
Golf          129
Equinox       128
Altima        127
F-150         127
Pilot         126
Highlander    123
Tahoe         120
X3            114
Q5            112
C-Class       110
Name: count, 

In [10]:
df = df.drop(df[df["Selling_Price"] == 500].index).reset_index(drop=True)

In [11]:
X=df.drop("Selling_Price", axis=1)
y=df["Selling_Price"]

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X, y, random_state=15, test_size=0.25)

In [13]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [14]:
cat_col=df.select_dtypes(include="object").columns.tolist()
num_col=df.select_dtypes(include=["int64","float64"]).columns.tolist()

In [15]:
cat_col

['Make',
 'Model',
 'Fuel_Type',
 'Transmission',
 'Service_History',
 'Color',
 'Body_Type',
 'Drivetrain',
 'Location']

In [16]:
num_col

['Year',
 'Engine_Size',
 'Mileage',
 'Horsepower',
 'Torque',
 'Owners',
 'Accident_History',
 'Fuel_Efficiency',
 'Selling_Price']

In [17]:
num_col.remove("Selling_Price")
num_col

['Year',
 'Engine_Size',
 'Mileage',
 'Horsepower',
 'Torque',
 'Owners',
 'Accident_History',
 'Fuel_Efficiency']

In [18]:
transformer=ColumnTransformer(
    transformers=[
        ("kategorik", OneHotEncoder(), cat_col),
        ("sayısal", StandardScaler(), num_col)
    ], remainder="passthrough"
)

X_train=transformer.fit_transform(X_train)
X_test=transformer.transform(X_test)

In [19]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 54621 stored elements and shape (3213, 94)>

In [20]:
from sklearn.linear_model import LinearRegression
linear=LinearRegression()

In [21]:
linear.fit(X_train, y_train)
y_pred=linear.predict(X_test)

In [22]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [23]:
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred))
print("mean_squared_error=", mean_squared_error(y_test, y_pred))
print("R2 Score=", r2_score(y_test, y_pred))

mean_absolute_error= 2739.252185773927
mean_squared_error= 14906931.54600622
R2 Score= 0.917469320073837


In [24]:
from sklearn.preprocessing import PolynomialFeatures
poly=PolynomialFeatures(degree=3)
X_train_poly=poly.fit_transform(X_train)
X_test_poly=poly.transform(X_test)
linear.fit(X_train_poly, y_train)
y_pred2=linear.predict(X_test_poly)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred2))
print("mean_squared_error=", mean_squared_error(y_test, y_pred2))
print("R2 Score=", r2_score(y_test, y_pred2))

mean_absolute_error= 2103.0661435212146
mean_squared_error= 9325155.799611868
R2 Score= 0.9483722423904495


In [25]:
from sklearn.linear_model import LassoCV
lasso=LassoCV(cv=5, random_state=15)
lasso.fit(X_train, y_train)
y_pred3=lasso.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred3))
print("mean_squared_error=", mean_squared_error(y_test, y_pred3))
print("R2 Score=", r2_score(y_test, y_pred3))

mean_absolute_error= 2755.735589087559
mean_squared_error= 15240407.10542613
R2 Score= 0.9156230672368434


In [26]:
from sklearn.linear_model import RidgeCV
ridge=RidgeCV(cv=5)
ridge.fit(X_train, y_train)
y_pred4=ridge.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred4))
print("mean_squared_error=", mean_squared_error(y_test, y_pred4))
print("R2 Score=", r2_score(y_test, y_pred4))

mean_absolute_error= 2739.1628905817847
mean_squared_error= 14942410.13443728
R2 Score= 0.9172728965498542


In [27]:
from sklearn.linear_model import ElasticNetCV
elastic=ElasticNetCV(cv=5, l1_ratio=[0.1, 0.5, 0.9], random_state=15)
elastic.fit(X_train, y_train)
y_pred5= elastic.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred5))
print("mean_squared_error=", mean_squared_error(y_test, y_pred5))
print("R2 Score=", r2_score(y_test, y_pred5))

mean_absolute_error= 6302.088191362013
mean_squared_error= 79992718.00788061
R2 Score= 0.5571286159088221


In [28]:
elastic.alphas_

array([[9.01056933e+04, 8.40328712e+04, 7.83693369e+04, 7.30875059e+04,
        6.81616526e+04, 6.35677854e+04, 5.92835294e+04, 5.52880179e+04,
        5.15617905e+04, 4.80866984e+04, 4.48458159e+04, 4.18233580e+04,
        3.90046037e+04, 3.63758239e+04, 3.39242151e+04, 3.16378366e+04,
        2.95055523e+04, 2.75169768e+04, 2.56624247e+04, 2.39328631e+04,
        2.23198683e+04, 2.08155838e+04, 1.94126831e+04, 1.81043333e+04,
        1.68841618e+04, 1.57462258e+04, 1.46849829e+04, 1.36952642e+04,
        1.27722492e+04, 1.19114424e+04, 1.11086510e+04, 1.03599651e+04,
        9.66173815e+03, 9.01056933e+03, 8.40328712e+03, 7.83693369e+03,
        7.30875059e+03, 6.81616526e+03, 6.35677854e+03, 5.92835294e+03,
        5.52880179e+03, 5.15617905e+03, 4.80866984e+03, 4.48458159e+03,
        4.18233580e+03, 3.90046037e+03, 3.63758239e+03, 3.39242151e+03,
        3.16378366e+03, 2.95055523e+03, 2.75169768e+03, 2.56624247e+03,
        2.39328631e+03, 2.23198683e+03, 2.08155838e+03, 1.941268

In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
svr_param = {
    "C": [100, 150, 200, 250],
    "kernel": ["linear", "poly", "rbf", "sigmoid"],
    "gamma": ["scale", "auto"]
}

grid_svr=GridSearchCV(estimator=SVR(), param_grid=svr_param, cv=5, scoring="r2", n_jobs=-1)

grid_svr.fit(X_train, y_train)
y_pred6= grid_svr.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred6))
print("mean_squared_error=", mean_squared_error(y_test, y_pred6))
print("R2 Score=", r2_score(y_test, y_pred6))

mean_absolute_error= 2772.052208819945
mean_squared_error= 22626664.429852292
R2 Score= 0.874729819876503


In [30]:
grid_svr.best_params_

{'C': 250, 'gamma': 'scale', 'kernel': 'linear'}

In [31]:
from sklearn.neighbors import KNeighborsRegressor

knr_param={
    "n_neighbors" : [3,5,7,9],
    "weights" : ["uniform", "distance"],
    "algorithm" : ["auto"],
    "metric" : ["euclidean", "manhattan", "minkowski"]
}

grid_knr=GridSearchCV(estimator=KNeighborsRegressor(), param_grid=knr_param, cv=5, scoring="r2", n_jobs=-1)

grid_knr.fit(X_train, y_train)
y_pred7=grid_knr.predict(X_test)
y_pred6= grid_svr.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred7))
print("mean_squared_error=", mean_squared_error(y_test, y_pred7))
print("R2 Score=", r2_score(y_test, y_pred7))

mean_absolute_error= 3895.1498296827235
mean_squared_error= 27311258.237314187
R2 Score= 0.8487940522831176


In [32]:
grid_knr.best_params_

{'algorithm': 'auto',
 'metric': 'manhattan',
 'n_neighbors': 9,
 'weights': 'distance'}

In [33]:
from sklearn.tree import DecisionTreeRegressor 

tree_param={
    "criterion" : ["squared_error", "absolute_error", "poisson"],
    "splitter" : ["best", "random"],
    "max_depth" : [5, 10, 15, None],
    "max_features" : [None, "sqrt", "log2"],
    "min_samples_split" : [3,5,10],
    "min_samples_leaf" : [1, 2, 4]
}

grid_tree=GridSearchCV(estimator=DecisionTreeRegressor(), cv=5, param_grid=tree_param, scoring="r2", n_jobs=-1 )
grid_tree.fit(X_train, y_train)
y_pred8= grid_tree.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred8))
print("mean_squared_error=", mean_squared_error(y_test, y_pred8))
print("R2 Score=", r2_score(y_test, y_pred8))

mean_absolute_error= 2488.405213343846
mean_squared_error= 10825776.522750964
R2 Score= 0.9400642114445946


In [34]:
grid_tree.best_params_

{'criterion': 'squared_error',
 'max_depth': 15,
 'max_features': None,
 'min_samples_leaf': 2,
 'min_samples_split': 10,
 'splitter': 'random'}

In [35]:
tree_param={
    "criterion" : ["poisson"],
    "splitter" : ["random"],
    "max_depth" : [None],
    "max_features" : [None],
    "min_samples_split" : [8, 10, 11, 12, 13, 14],
    "min_samples_leaf" : [2, 3]
}

grid_tree=GridSearchCV(estimator=DecisionTreeRegressor(), cv=5, param_grid=tree_param, scoring="r2", n_jobs=-1 )
grid_tree.fit(X_train, y_train)
y_pred9= grid_tree.predict(X_test)
print("mean_absolute_error=", mean_absolute_error(y_test, y_pred9))
print("mean_squared_error=", mean_squared_error(y_test, y_pred9))
print("R2 Score=", r2_score(y_test, y_pred9))

mean_absolute_error= 2457.688032990974
mean_squared_error= 11366114.421974892
R2 Score= 0.937072686725024


In [36]:
grid_tree.best_params_

{'criterion': 'poisson',
 'max_depth': None,
 'max_features': None,
 'min_samples_leaf': 2,
 'min_samples_split': 8,
 'splitter': 'random'}

---
## 📊 Sonuçların Karşılaştırılması

| Model | MAE | MSE | R² Skoru |
|---|---|---|---|
| Linear Regression | ~3799 | ~26.2M | 0.8435 |
| Polynomial Regression (derece 3) | ~1876 | ~7.96M | 0.9524 |
| Lasso (CV) | ~3799 | ~26.3M | 0.8428 |
| Ridge (CV) | ~3798 | ~26.2M | 0.8435 |
| ElasticNet (CV) | ~5526 | ~65.5M | 0.6089 |
| SVR (GridSearch) | ~3572 | ~31.4M | 0.8122 |
| KNN (GridSearch) | ~3005 | ~22.3M | 0.8670 |
| Decision Tree (GridSearch) | ~1851 | ~8.12M | 0.9515 |
| Decision Tree (2. GridSearch) | ~2006 | ~9.37M | 0.9440 |

> Not: Aykırı değer temizliği sonrası veri seti değiştiği için, notebook'u baştan sona çalıştırdığında bu skorlar güncellenecektir.

## ✅ Sonuç ve Değerlendirme
- En yüksek performansı **Polynomial Regression (derece 3)** ve **Decision Tree Regressor** modelleri göstermiştir (R² ≈ 0.95).
- Doğrusal modeller (Linear, Lasso, Ridge) benzer ve daha düşük performans sergilemiştir; bu da veri setinde **doğrusal olmayan ilişkiler** bulunduğuna işaret etmektedir.
- ElasticNet, düzenlileştirme (regularization) etkisiyle en düşük performansı göstermiştir.
- SVR ve KNN, orta seviye bir performans sunmuştur.
